In [1]:
import pandas as pd

Вам будут предоставлены данные и ipynb-файл, который представляет собой анализ этих данных с целью выявления необходимых признаков, на которых в будущем будет построена модель. Мы рассмотрим имеющиеся вещественные/бинарные признаки и обсудим, какие простейшие признаки можно вытащить из остальных колонок. В ноутбуке вы можете найти описание данных и ключевые точки исследования.

Датасет и ipynb-файл можно скачать здесь: taxi_dataset_with_predictions, taxi.ipynb

Представим, что 2 добрых и очень-очень умных гнома прокрались к нам в данные, описывающие различные поездки в такси.
Далее, для каждого объекта (и его признаков, соответственно), гномы построили модели предсказания того, какая должна была быть у данных заказов длительность исполнения. Прогнозы своих моделей они положили соответственно в колонки **prediction_1** и **prediction_2**

In [2]:
df = pd.read_csv('taxi_dataset_with_predictions.csv', index_col=0)

In [3]:
df.head()

,vendor_id,pickup_datetime,passenger_count,store_and_fwd_flag,trip_duration,distance_km,prediction_1,prediction_2
id,,,,,,,,
id2875421,1,2016-03-14 17:24:55,930.399753,0,455.0,1.500479,578.156451,355.270710
id2377394,0,2016-06-12 00:43:35,930.399753,0,663.0,1.807119,962.657188,674.295781
id3858529,1,2016-01-19 11:35:24,930.399753,0,2124.0,6.392080,2546.180515,2422.132431
id3504673,1,2016-04-06 19:32:31,930.399753,0,429.0,1.487155,737.926214,795.992362
id2181028,1,2016-03-26 13:30:55,930.399753,0,435.0,1.189925,666.070794,-4.158492


В начале посчитайте **MSE** модели. Для этого нужно посчитать квадратичное отклонение на каждом объекте, а потом просто усредниться! Полезно иметь формулу перед глазами.

$$
MSE = \frac{1}{n} \sum_i^n (a(x_i)-y_i)^2
$$

In [4]:
### Your code is here

error_1 = ((df['prediction_1'] - df['trip_duration'])**2).mean()
error_2 = ((df['prediction_2'] - df['trip_duration'])**2).mean()

print(f"MSE первой модели равно: {int(error_1)}")
print(f"MSE второй модели равно: {int(error_2)}")

MSE первой модели равно: 99994
MSE второй модели равно: 124936


Видно, что у MSE достаточно большой порядок. Как мы и говорили, глазам куда будет приятнее, если мы будем считать **RMSE**:

$$
RMSE = \sqrt{MSE} = \sqrt{\frac{1}{n} \sum_i^n (a(x_i)-y_i)^2}|
$$

In [5]:
### Your code is here

print(f"RMSE первой модели равно: {int(error_1**(1/2))}")
print(f"RMSE второй модели равно: {int(error_2**(1/2))}")

RMSE первой модели равно: 316
RMSE второй модели равно: 353


Давайте теперь замерим значения средней абсолютной ошибки, то есть **MAE**:

$$
MAE = \frac{1}{n} \sum_i^n |a(x_i)-y_i|
$$

In [6]:
### Your code is here

absolute_error_1 = (abs(df['prediction_1'] - df['trip_duration'])).mean()
absolute_error_2 = (abs(df['prediction_2'] - df['trip_duration'])).mean()

print(f"MAE первой модели равно: {int(absolute_error_1)}")
print(f"MAE второй модели равно: {int(absolute_error_2)}")

MAE первой модели равно: 300
MAE второй модели равно: 281


Ситуация, когда **MAE** и **MSE**, выбирая между 2-х,  указывают на разные модели, знакома нам еще из лекции. 

Такое может происходить, когда в одной из моделей ошибка, в среднем, независимо от порядка чисел, чуть-чуть получше, чем во второй. Но при этом если первая модель и ошибается, то куда суровее второй. 

Представьте: Петя и Миша играют в дартс. Петя в 9/10 случаев попадает в яблочко, но каждый 10-ый раз кидает дротик в потолок. Миша же, хоть и не так часто забрасывает в центр (всего 6/10), но при этом, оставшиеся 4 попытки реализует не совсем фатально: попадает всего-лишь немного дальше яблочка. 

В таком выдуманном сценарии Петя будет лучше с точки зрения **MAE**, а Миша - с точки зрения **MSE**, так как не допускает фатальных ошибок, хоть и в среднем набирает меньше очков. 

Давайте и вправду убедимся в том, что фатальных ошибок у второй модели больше. 

Посчитайте, в скольки случаях предсказания откланяются от ответа более, чем на **500**, для первой и второй моделей!

In [ ]:
### Your code is here

counter_1 = sum(abs(df['prediction_1'] - df['trip_duration']) > 500)
counter_2 = sum(abs(df['prediction_2'] - df['trip_duration']) > 500)

print(f"Количество отклонений > 500 от верного ответа для первой модели равно: {counter_1}")
print(f"Количество отклонений > 500 от верного ответа для второй модели равно: {counter_2}")

Полученные результаты кажутся нам логичными: как и подмечали ранее, первая модель скорее лучше в смысле "суровых ошибок", когда у второй среднее абсолюнтное отклонение прогноза от действительности поменьше.

Рассчитайте **RMSLE**. Придумайте, как проверить характер несимметричности данной метрики: за что она наказывает больше - за перепредсказания или за недопредсказания? Это нужно будет в тестовом теоретическом задании.

$$
\text{RMSLE}(X, y, a) = \sqrt{\frac{1}{\ell}\sum_{i=1}^{\ell} \big(\log{(y_i + 1)} - \log{(a(x_i) + 1)}\big)^2}
$$

Для взятия логарифма используйте библиотеку **math**


In [19]:
### Your code is here

df['prediction_1'] = df['prediction_1'].apply(lambda x: max(x, 0))
df['prediction_2'] = df['prediction_2'].apply(lambda x: max(x, 0))

In [ ]:
from math import log

under_estimating = ((log(90+1)-log(100+1))**2)**(1/2)

over_estimating = ((log(110+1)-log(100+1))**2)**(1/2)

print(f"Ошибка RMSLE при недопредсказании: {round(under_estimating, 3)}")
print(f"Ошибка RMSLE при перепредсказании: {round(over_estimating, 3)}")

Ошибка RMSLE при недопредсказании: 0.104
Ошибка RMSLE при перепредсказании: 0.094


Как видно, RMSLE больше штрафует за недопредсказание!

In [21]:
from math import log


msle_1 = (((df['prediction_1'] + 1).apply(log) - (df['trip_duration'] + 1).apply(log))**2).mean()
msle_2 = (((df['prediction_2'] + 1).apply(log) - (df['trip_duration'] + 1).apply(log))**2).mean()

print(f"RMSLE первой модели равно: {msle_1**(1/2)}")
print(f"RMSLE второй модели равно: {msle_2**(1/2)}")

RMSLE первой модели равно: 0.5537581774590482
RMSLE второй модели равно: 1.5564340528341787


Рассчитайте RMSLE. Придумайте, как проверить характер несимметричности данной метрики: за что она наказывает больше - за перепредсказания или за недопредсказания? Это нужно будет в тестовом теоретическом задании.

В ответ введите значение RMSLE сначала для первой, а затем для второй модели через запятую без пробела округленное до 3-го знака после точки.

Посчитайте, для какого количества объектов первая модель сделала перепредсказания и недопредсказания. В ответ введите значение сначала для случая, когда предсказания оказались больше действительных, а затем меньше действительных через запятую без пробела.

### Задание 9

In [22]:
### Your code is here

over_predicted_1 = sum(df['prediction_1'] - df['trip_duration'] > 0)

under_predicted_1 = sum(df['prediction_1'] - df['trip_duration'] < 0)

print(f"Предсказания первой модели оказались больше действительных в {over_predicted_1} случаях")
print(f"Предсказания первой модели оказались меньше действительных в {under_predicted_1} случаях")

Предсказания первой модели оказались больше действительных в 1456721 случаях
Предсказания первой модели оказались меньше действительных в 1923 случаях


Посчитайте, для какого количества объектов вторая модель сделала перепредсказания и недопредсказания. В ответ введите значение сначала для случая, когда предсказания оказались больше действительных, а затем меньше действительных через запятую без пробела.

### Задание 10

In [23]:
### Your code is here

over_predicted_2 = sum(df['prediction_2'] - df['trip_duration'] > 0)

under_predicted_2 = sum(df['prediction_2'] - df['trip_duration'] < 0)

print(f"Предсказания второй модели оказались больше действительных в {over_predicted_2} случаях")
print(f"Предсказания второй модели оказались меньше действительных в {under_predicted_2} случаях")

Предсказания второй модели оказались больше действительных в 811778 случаях
Предсказания второй модели оказались меньше действительных в 646866 случаях
